# 01 - Bench capture overview

**Question:** what do the first O4/ELRS bench captures look like, how do
they compare against their MATCHED TX-OFF controls, and is the ELRS duty
cycle stable in time or lumpy block-by-block?

**Scope (receive-only):** HackRF recordings of bench transmitters
(RadioMaster Boxer on ELRS 2.4 GHz). Everything here is analysis of
received data: nothing transmits, jams or otherwise interferes.

**Matched controls:** the corpus contains TX-OFF captures
(`protocol: "noise"`) at the same center frequency and gain as the signal
captures. SNR and duty thresholds are computed against the MATCHED control
of each signal (pairing table printed below, ids included) — never against
a different window or gain. Captures without a matched control fall back
to the labeled legacy -35 dB relative threshold.

**SYNTHETIC fallback:** with no entries under
`purpose = signature_capture`, an in-memory synthetic signal + matched
synthetic control exercise the whole pipeline. All synthetic output is
labeled SYNTHETIC — it is NOT measurements.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal

from tacet.dsp import features
from tacet.loaders import cs8
from tacet.loaders import manifest as manifest_mod

# --- Configuration --------------------------------------------------------
SEGMENT_S = 2.0           # features / SNR segment (first N seconds)
NFFT = 1024               # spectrogram nfft (duty envelope, plots)
SNR_NFFT = 2048           # Welch nfft for the per-bin SNR (same for both)
SNR_MIN_RATIO_DB = 3.0    # a bin is "occupied" if signal > control + 3 dB
THRESHOLD_REL_DB = -35.0  # LEGACY fallback threshold (no matched control)
QC_BLOCK = 10_000_000     # complex samples per streamed QC block
DUTY_CHUNK_S = 1.0        # duty series: spectrogram per 1 s chunk
DUTY_BLOCK_S = 0.1        # duty series: block size in seconds
DUTY_CONTROL_PCTL = 95.0  # threshold = this percentile of control envelope
SYNTHETIC_FALLBACK = True # demo pipeline when the manifest has no captures
print("config ok")


In [ ]:
# --- Load: signals + controls, matched by (center_freq, total gain) --------
def gain_key(e):
    g = e["gain"]
    return (g.get("lna_db", 0) or 0) + (g.get("vga_db", 0) or 0)

def freq_key(e):
    return round(float(e["center_freq_hz"]))

rows = manifest_mod.query(purpose="signature_capture") if Path("manifest.json").exists() else []
signals = [e for e in rows if e["protocol"] != "noise"]
controls = [e for e in rows if e["protocol"] == "noise"]
controls_by_key = {}
for c in controls:
    controls_by_key.setdefault((freq_key(c), gain_key(c)), []).append(c)

# synthetic fallback: one band-limited burst signal + its matched CONTROL
# (TX-OFF: noise floor + DC only, no bursts, no clipping). In memory only.
if not rows and SYNTHETIC_FALLBACK:
    fs = 1_000_000.0
    dur_s = 2.0
    n = int(fs * dur_s)
    t = np.arange(n) / fs
    rng = np.random.default_rng(11)
    base = rng.standard_normal(n) + 1j * rng.standard_normal(n)
    taps = signal.firwin(255, 100e3, fs=fs)
    tt = np.arange(taps.size) / fs
    bc = taps * np.exp(2j * np.pi * 150e3 * tt)
    band = signal.lfilter(bc, 1.0, base)
    burst_gate = (np.mod(t, 0.5) < 0.15).astype(float)
    sig = (band * burst_gate + 50.0).astype(np.complex64)
    ctrl = (1e-3 * base + 50.0).astype(np.complex64)   # TX-OFF: floor + DC
    signals = [{"id": "synthetic_bench", "path": None, "z": sig, "fs": fs,
                "protocol": "elrs", "notes": "SYNTHETIC in-memory demo",
                "synthetic": True}]
    controls = [{"id": "synthetic_control", "path": None, "z": ctrl,
                 "fs": fs, "protocol": "noise",
                 "notes": "SYNTHETIC matched control", "synthetic": True}]
    print("SYNTHETIC fallback — NOT measurements")

captures = []
for sig in signals:
    key = (freq_key(sig), gain_key(sig))
    cands = controls_by_key.get(key, [])
    sig = dict(sig, control_candidates=cands, control=None)
    captures.append(sig)
print(f"signals: {len(captures)}, controls: {len(controls)}")
for cap in captures:
    ids = [c["id"] for c in cap["control_candidates"]]
    state = "matched" if ids else "NO MATCHED CONTROL (legacy fallback)"
    print(f"  {cap['id']}  ({freq_key(cap)/1e6:.0f} MHz, +{gain_key(cap)} dB) "
          f"-> {state}: {ids}")


In [ ]:
# --- QC: rail_fraction streamed over the WHOLE capture ----------------------
# Applied to signals AND controls (a saturated control is a bad reference).
for cap in captures + controls:
    if cap["path"] is not None:
        n_rail, n_total = 0.0, 0
        for block in cs8.iter_cs8_blocks(cap["path"], QC_BLOCK):
            n_rail += features.rail_fraction(block) * block.size
            n_total += block.size
        cap["rail"] = n_rail / n_total if n_total else 0.0
    else:
        cap["rail"] = features.rail_fraction(cap["z"])
    flag = "RAIL QC FAIL (> 0.1%) - invalid capture" if cap["rail"] > 1e-3 else "rail qc ok"
    print(f"{cap['id']}: rail_fraction(whole) = {cap['rail']:.6%}  [{flag}]")


In [ ]:
# --- Feature table (first SEGMENT_S per capture) ----------------------------
qc_rows = []
for cap in captures + controls:
    if cap["path"] is not None:
        z = cs8.load_cs8(cap["path"], n_samples=int(cap["fs"] * SEGMENT_S))
    else:
        z = cap["z"][: int(cap["fs"] * SEGMENT_S)]
    y = features.remove_dc(z)
    f_w, p = features.welch_psd(y, cap["fs"], nfft=NFFT)
    bw, f_lo, f_hi = features.occupied_bandwidth(f_w, p, percent=99.0)
    f_s, t_s, S = features.spectrogram(y, cap["fs"], nfft=NFFT)
    env = features.time_envelope(S)
    duty = features.duty_cycle(env, threshold_rel_db=THRESHOLD_REL_DB)
    bursts = features.burst_structure(env, t_s, threshold_rel_db=THRESHOLD_REL_DB)
    del y, S, env   # heavy arrays released; nothing else is retained
    qc_rows.append({
        "capture": cap["id"],
        "protocol": cap["protocol"],
        "synthetic": cap["synthetic"],
        "rail_fraction_whole": cap["rail"],
        "flatness": features.spectral_flatness(p),
        "obw_hz": bw,
        "duty_legacy_-35dB": duty,
        "n_bursts": len(bursts["bursts"]),
    })

df = pd.DataFrame(qc_rows)
df


In [ ]:
# --- Matched-control SNR ----------------------------------------------------
# (a) window-mean PSD ratio over the whole segment: a LOWER BOUND for FHSS,
#     because averaging dilutes dispersed hops.
# (b) per-bin SNR: occupied bins = signal bins exceeding the control's
#     per-bin PSD by > SNR_MIN_RATIO_DB; report the occupied-bin fraction
#     and the MEDIAN per-bin ratio (dB) over those bins. Same nfft and the
#     same segment length for signal and control.
# The control id used is PRINTED for every signal — never silent. Captures
# without a matched control skip the comparison with a clear note.
def pick_control(cap):
    cands = [c for c in cap["control_candidates"] if c["rail"] < 1e-3]
    if not cands:
        cands = cap["control_candidates"]
    return cands[0] if cands else None

def segment_for(cap, n_samples):
    if cap["path"] is not None:
        z = cs8.load_cs8(cap["path"], n_samples=n_samples)
    else:
        z = cap["z"][:n_samples]
    return features.remove_dc(z)

snr_rows = []
for cap in captures:
    ctrl = pick_control(cap)
    cap["control"] = ctrl
    if ctrl is None:
        print(f"{cap['id']}: no matched control at "
              f"({freq_key(cap)/1e6:.0f} MHz, +{gain_key(cap)} dB) — "
              "SNR comparison SKIPPED (legacy -35 dB threshold applies "
              "to the duty cell)")
        continue
    n_seg = int(cap["fs"] * SEGMENT_S)
    y_sig = segment_for(cap, n_seg)
    y_ctrl = segment_for(ctrl, n_seg)
    m = min(y_sig.size, y_ctrl.size)
    y_sig, y_ctrl = y_sig[:m], y_ctrl[:m]   # same length for both
    _, p_sig = features.welch_psd(y_sig, cap["fs"], nfft=SNR_NFFT)
    _, p_ctrl = features.welch_psd(y_ctrl, cap["fs"], nfft=SNR_NFFT)
    snr_lb_db = 10.0 * np.log10(float(np.mean(p_sig)) / float(np.mean(p_ctrl)))
    ratio_db = 10.0 * np.log10(p_sig / np.maximum(p_ctrl, 1e-300))
    occ = ratio_db > SNR_MIN_RATIO_DB
    occ_frac = float(np.mean(occ))
    med_occ_db = float(np.median(ratio_db[occ])) if occ.any() else float("nan")
    print(f"{cap['id']} vs control {ctrl['id']}:")
    print(f"  SNR lower bound (window-mean) : {snr_lb_db:+.1f} dB")
    print(f"  occupied bins (> +{SNR_MIN_RATIO_DB:.0f} dB/bin)    : "
          f"{occ_frac:.1%} of {ratio_db.size}")
    print(f"  median per-bin ratio (occupied): {med_occ_db:+.1f} dB")
    snr_rows.append({"capture": cap["id"], "control": ctrl["id"],
                     "snr_lower_bound_db": snr_lb_db,
                     "occupied_fraction": occ_frac,
                     "median_occupied_bin_ratio_db": med_occ_db})

snr_df = pd.DataFrame(snr_rows)
snr_df


In [ ]:
# --- Duty time-series per ~100 ms block over the FULL capture ---------------
# Envelopes are computed per 1 s chunk (no full-capture spectrogram in
# memory). The threshold is DERIVED from the matched control: 95th
# percentile of the control's own envelope — no magic constants. Captures
# without a matched control fall back to the labeled legacy -35 dB
# relative threshold.
def envelope_chunks(chunks, fs):
    envs = []
    for chunk in chunks:
        _, _, S = features.spectrogram(chunk, fs, nfft=NFFT)
        envs.append(features.time_envelope(S))
        del S
    return np.concatenate(envs) if envs else np.array([])

def file_chunks(path, fs):
    for block in cs8.iter_cs8_blocks(path, int(fs)):
        yield block

def array_chunks(z, fs):
    for i in range(0, z.size, int(fs)):
        yield z[i: i + int(fs)]

def duty_series(env, thr, fs_env, block_s=DUTY_BLOCK_S):
    n_block = max(int(round(block_s * fs_env)), 1)
    n_full = (env.size // n_block) * n_block
    blocks = env[:n_full].reshape(-1, n_block)
    centers = (np.arange(blocks.shape[0]) + 0.5) * block_s
    return centers, (blocks >= thr).mean(axis=1)

duty_stats = []
for cap in captures:
    fs = cap["fs"]
    fs_env = fs / NFFT
    ctrl = cap.get("control")
    chunks = (file_chunks(cap["path"], fs) if cap["path"] is not None
              else array_chunks(cap["z"], fs))
    env = envelope_chunks(chunks, fs)
    if ctrl is not None:
        ctrl_chunks = (file_chunks(ctrl["path"], fs) if ctrl["path"] is not None
                       else array_chunks(ctrl["z"], fs))
        ctrl_env = envelope_chunks(ctrl_chunks, fs)
        thr = float(np.percentile(ctrl_env, DUTY_CONTROL_PCTL))
        thr_src = (f"95th percentile of control {ctrl['id']} envelope")
    else:
        ctrl_env = None
        thr = float(env.max() * 10.0 ** (THRESHOLD_REL_DB / 10.0))
        thr_src = f"LEGACY fallback: peak {THRESHOLD_REL_DB} dB (no control)"
    centers, duty = duty_series(env, thr, fs_env)
    ctrl_centers, ctrl_duty = (None, None)
    if ctrl_env is not None:
        ctrl_centers, ctrl_duty = duty_series(ctrl_env, thr, fs_env)
        ctrl_self = float(np.mean(ctrl_env >= thr))
        print(f"{cap['id']}: control's own duty vs its own threshold = "
              f"{ctrl_self:.1%} (~{100 - DUTY_CONTROL_PCTL:.0f}% by construction)")
    print(f"{cap['id']}: threshold {thr:.3g} [{thr_src}]")
    print(f"  block duty over {duty.size} x {DUTY_BLOCK_S*1e3:.0f} ms blocks: "
          f"mean {duty.mean():.1%} +/- {duty.std():.1%}")
    cap["duty_mean"], cap["duty_std"] = float(duty.mean()), float(duty.std())
    duty_stats.append(cap)
    label = cap["id"] + (" (SYNTHETIC)" if cap["synthetic"] else "")
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(centers, duty * 100, "o-", ms=3, label="signal")
    if ctrl_duty is not None:
        ax.plot(ctrl_centers, ctrl_duty * 100, "s--", ms=3,
                label=f"control {ctrl['id'][:20]}")
    ax.set_xlabel("t [s]")
    ax.set_ylabel("block duty [%] (>= threshold)")
    ax.set_title(f"block duty series, {label}")
    ax.set_ylim(-2, 102)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


In [ ]:
# --- Verdict: does the 2450 quiet vs 2450 re-scan difference persist
# --- block-by-block (temporal clustering)? ---------------------------------
windows_2450 = sorted(
    [c for c in duty_stats
     if freq_key(c) == 2_450_000_000 and c["protocol"] != "noise"],
    key=lambda c: c["id"])
if len(windows_2450) >= 2:
    quiet, rescan = windows_2450[0], windows_2450[-1]
    m_q, s_q = quiet["duty_mean"], quiet["duty_std"]
    m_r, s_r = rescan["duty_mean"], rescan["duty_std"]
    separated = abs(m_r - m_q) > (s_q + s_r)
    print(f"quiet   {quiet['id']}: block duty {m_q:.1%} +/- {s_q:.1%} "
          f"({quiet['notes'][:40]})")
    print(f"re-scan {rescan['id']}: block duty {m_r:.1%} +/- {s_r:.1%} "
          f"({rescan['notes'][:40]})")
    if separated:
        print("VERDICT: the re-scan's higher duty PERSISTS block-by-block "
              "(mean separation exceeds the block-to-block spread): the "
              "duty difference is temporal clustering of the hopping, not "
              "measurement noise. Detector integration time must span "
              "multiple 100 ms blocks.")
    else:
        print("VERDICT: the re-scan vs quiet difference does NOT clearly "
              "persist block-by-block (means within the block-to-block "
              "spread): no evidence of stable temporal clustering in this "
              "pair; treat the earlier duty delta as not yet established.")
else:
    print("need two 2450 MHz signal captures for the lumpiness comparison")


In [ ]:
# --- Side-by-side: signal vs matched control (first SEGMENT_S) --------------
def db_spectrogram_for(cap, seg_s):
    n = int(cap["fs"] * seg_s)
    if cap["path"] is not None:
        z = cs8.load_cs8(cap["path"], n_samples=n)
    else:
        z = cap["z"][:n]
    f, t, S = features.spectrogram(features.remove_dc(z), cap["fs"], nfft=NFFT)
    db = 10.0 * np.log10(S + 1e-30)
    del S
    return f, t, db

for cap in captures:
    ctrl = cap.get("control")
    if ctrl is None:
        continue
    f1, t1, db_sig = db_spectrogram_for(cap, SEGMENT_S)
    f2, t2, db_ctrl = db_spectrogram_for(ctrl, SEGMENT_S)
    vmin = min(db_sig.min(), db_ctrl.min())
    vmax = max(db_sig.max(), db_ctrl.max())
    label = " (SYNTHETIC)" if cap["synthetic"] else ""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    im1 = ax1.pcolormesh(t1, f1 / 1e6, db_sig, shading="auto",
                         cmap="viridis", vmin=vmin, vmax=vmax)
    ax1.set_title(f"signal {cap['id']}{label}")
    im2 = ax2.pcolormesh(t2, f2 / 1e6, db_ctrl, shading="auto",
                         cmap="viridis", vmin=vmin, vmax=vmax)
    ax2.set_title(f"control {ctrl['id']}")
    for ax, im in ((ax1, im1), (ax2, im2)):
        fig.colorbar(im, ax=ax, label="dB")
        ax.set_xlabel("t [s]")
    ax1.set_ylabel("f [MHz]")
    fig.suptitle(f"{freq_key(cap)/1e6:.0f} MHz — signal vs matched control "
                 f"(shared color scale)")
    fig.tight_layout()
    plt.show()


## Interpretation guide

- **SNR as a lower bound:** for a frequency-hopping link the window-mean
  PSD ratio over 20 MHz dilutes the hops, so the window-mean figure is a
  LOWER BOUND on the SNR. The per-bin view is the honest one: occupied
  bins (signal > control + 3 dB) hold the hops; the median per-bin ratio
  over those bins is what a detector channel would see. Both are reported,
  labeled, per signal, against its MATCHED control (same window, same
  gain) — the control id is printed with every comparison.
- **Duty threshold derivation:** the block-duty threshold is the 95th
  percentile of the matched control's own envelope (computed with the
  same chunked pipeline). No magic constants: whatever the control's
  envelope distribution looks like, ~5% of the control's own blocks sit
  above its threshold by construction (printed per capture). Captures
  without a matched control fall back to the legacy peak -35 dB
  threshold, clearly labeled.
- **Duty lumpiness (the deliverable):** the block-duty series covers the
  FULL recording in ~100 ms blocks (not a 2 s snippet). If the 2450 MHz
  re-scan's duty stays far above the quiet window's across blocks, the
  ELRS hopping duty is temporally clustered — the earlier 5.4% -> 33.6%
  delta is real structure, not noise — and detector integration time must
  span several 100 ms blocks. The verdict cell states the measured
  conclusion either way.
- **Rail QC:** `rail_fraction` over the whole capture; above 0.1% the
  capture is invalid (drop LNA/VGA and re-record). A saturated control is
  rejected as a reference automatically.
- **DC spike:** `remove_dc` plus the DC-bin exclusion inside
  `time_envelope`/`spectral_flatness` keep the HackRF center spike out of
  every power statistic.
